In [1]:
import pandas, pyarrow, duckdb
print(pandas.__version__, pyarrow.__version__, duckdb.__version__)

3.1.1 26.0.1 2.0.0


In [2]:
import pandas as pd
from pathlib import Path

# Root repo = satu level di atas folder notebooks
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CLEAN = ROOT / "data" / "clean"

# Tampilkan semua nama file parquet (untuk memastikan nama lengkapnya)
for f in sorted(CLEAN.glob("*.parquet")):
    print(f.name)

bps_population_kapanewon_diy_2021_2022.parquet
coverage_kapanewon_hourly.parquet
dim_kapanewon.parquet
home_maids_vs_population_kapanewon.parquet
od_home_visits_kapanewon_monthly.parquet
od_origin_homes_monthly.parquet
od_summary.parquet
presence_cell500m_daily.parquet
presence_kapanewon_hourly.parquet
tier1_summary.parquet
validation_night_presence_kapanewon.parquet


In [3]:
for f in sorted(CLEAN.glob("*.parquet")):
    df = pd.read_parquet(f)
    print("=" * 70)
    print(f"{f.stem}  | shape = {df.shape}")
    print(df.dtypes.to_string())
    print(df.head(3).to_string())

bps_population_kapanewon_diy_2021_2022  | shape = (156, 10)
kapanewon_code            str
kab_kota               string
kapanewon              string
tahun                   int16
laki_laki               Int64
perempuan               Int64
total                   Int64
total_published         Int64
flag_total_mismatch      bool
kab_code                  str
  kapanewon_code     kab_kota kapanewon  tahun  laki_laki  perempuan  total  total_published  flag_total_mismatch kab_code
0       34.01.01  KULON PROGO     TEMON   2021      14159      14278  28437            28437                False    34.01
1       34.01.02  KULON PROGO     WATES   2021      24317      24571  48888            48888                False    34.01
2       34.01.03  KULON PROGO  PANJATAN   2021      18990      19395  38385            38385                False    34.01
coverage_kapanewon_hourly  | shape = (11451, 6)
kapanewon_code               str
date              datetime64[ns]
jam_terbit                 int64
k

In [4]:
import pyarrow.parquet as pq
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CLEAN = ROOT / "data" / "clean"

for f in sorted(CLEAN.glob("*.parquet")):
    pf = pq.ParquetFile(f)
    schema = pf.schema_arrow
    print(f"\n### {f.stem} | baris = {pf.metadata.num_rows:,}")
    print(", ".join(f"{n}:{t}" for n, t in zip(schema.names, schema.types)))


### bps_population_kapanewon_diy_2021_2022 | baris = 156
kapanewon_code:string, kab_kota:string, kapanewon:string, tahun:int16, laki_laki:int64, perempuan:int64, total:int64, total_published:int64, flag_total_mismatch:bool, kab_code:string

### coverage_kapanewon_hourly | baris = 11,451
kapanewon_code:string, date:timestamp[ns], jam_terbit:int64, kab_kota:string, kapanewon:string, jam_terbit_pct:double

### dim_kapanewon | baris = 78
kapanewon_code:string, kab_kota:string, kapanewon:string, kab_code:string

### home_maids_vs_population_kapanewon | baris = 78
kapanewon_code:string, kab_code:string, kab_kota:string, kapanewon:string, n_users:int32, pop_actual:double, ratio:double, ratio_gt_1:bool

### od_home_visits_kapanewon_monthly | baris = 9,894
month:timestamp[ns], origin_kab_code:string, origin_kab_kota:string, origin_code:string, origin_kapanewon:string, dest_kab_code:string, dest_kab_kota:string, dest_code:string, dest_kapanewon:string, n_maids:int32, reduced_quality:bool, is_in

In [5]:
import pandas as pd

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 50)

print(pd.read_parquet(CLEAN / "dim_kapanewon.parquet").head(5))
print(pd.read_parquet(CLEAN / "presence_kapanewon_hourly.parquet").head(5))

  kapanewon_code     kab_kota kapanewon kab_code
0       34.01.01  KULON PROGO     TEMON    34.01
1       34.01.02  KULON PROGO     WATES    34.01
2       34.01.03  KULON PROGO  PANJATAN    34.01
3       34.01.04  KULON PROGO     GALUR    34.01
4       34.01.05  KULON PROGO    LENDAH    34.01
  kapanewon_code kab_code     kab_kota kapanewon            datetime       date  hour  dayofweek  is_weekend    month  n_maids  n_pings  pings_per_maid  share_in_hour supply_regime  is_anomaly_day
0       34.01.01    34.01  KULON PROGO     TEMON 2021-12-01 00:00:00 2021-12-01     0          2       False  2021-12       16       66        4.125000       0.004207             A           False
1       34.01.01    34.01  KULON PROGO     TEMON 2021-12-01 01:00:00 2021-12-01     1          2       False  2021-12        8       38        4.750000       0.002566             A           False
2       34.01.01    34.01  KULON PROGO     TEMON 2021-12-01 02:00:00 2021-12-01     2          2       False  2021-

In [6]:
import pandas as pd
import duckdb
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CLEAN = ROOT / "data" / "clean"
OUT = ROOT / "data" / "integrated"
OUT.mkdir(parents=True, exist_ok=True)

dim    = pd.read_parquet(CLEAN / "dim_kapanewon.parquet")
bps    = pd.read_parquet(CLEAN / "bps_population_kapanewon_diy_2021_2022.parquet")
hourly = pd.read_parquet(CLEAN / "presence_kapanewon_hourly.parquet")
hm     = pd.read_parquet(CLEAN / "home_maids_vs_population_kapanewon.parquet")
val    = pd.read_parquet(CLEAN / "validation_night_presence_kapanewon.parquet")

print("dim", dim.shape, "| bps", bps.shape, "| hourly", hourly.shape, "| hm", hm.shape, "| val", val.shape)
print("Periode hourly:", hourly["date"].min(), "->", hourly["date"].max())
print("Nilai period di val:", val["period"].unique())

dim (78, 4) | bps (156, 10) | hourly (230418, 16) | hm (78, 8) | val (156, 9)
Periode hourly: 2021-12-01 00:00:00 -> 2022-04-30 00:00:00
Nilai period di val: <ArrowStringArray>
['Dec 2021 - Apr 2022', 'Dec 2021 - Jan 2022']
Length: 2, dtype: string


In [7]:
# 1. Key harus unik
print("dim: kode duplikat      =", dim["kapanewon_code"].duplicated().sum())
print("bps: (kode,tahun) dupl. =", bps.duplicated(["kapanewon_code", "tahun"]).sum())
print("hm : kode duplikat      =", hm["kapanewon_code"].duplicated().sum())

# 2. Tes JETIS: harus 2 baris dengan kode berbeda
print(dim[dim["kapanewon"].str.upper() == "JETIS"])

# 3. Anti-join dua arah: kode di satu tabel tapi tidak ada di tabel lain
kode_dim    = set(dim["kapanewon_code"])
kode_hourly = set(hourly["kapanewon_code"])
kode_bps    = set(bps["kapanewon_code"])
kode_hm     = set(hm["kapanewon_code"])

print("\nhourly - dim :", sorted(kode_hourly - kode_dim))
print("dim - hourly :", sorted(kode_dim - kode_hourly))
print("bps - dim    :", sorted(kode_bps - kode_dim))
print("dim - bps    :", sorted(kode_dim - kode_bps))
print("hm - dim     :", sorted(kode_hm - kode_dim))
print("dim - hm     :", sorted(kode_dim - kode_hm))

dim: kode duplikat      = 0
bps: (kode,tahun) dupl. = 0
hm : kode duplikat      = 0
   kapanewon_code    kab_kota kapanewon kab_code
20       34.02.09      BANTUL     JETIS    34.02
65       34.71.02  YOGYAKARTA     JETIS    34.71

hourly - dim : []
dim - hourly : []
bps - dim    : []
dim - bps    : []
hm - dim     : []
dim - hm     : []


In [8]:
hourly["tahun"] = hourly["date"].dt.year.astype("int16")

bps_join = bps[["kapanewon_code", "tahun", "laki_laki", "perempuan", "total",
                "flag_total_mismatch"]].rename(
    columns={"total": "pop_bps", "laki_laki": "pop_bps_laki", "perempuan": "pop_bps_perempuan"})

n_before = len(hourly)
integrated_hourly = hourly.merge(
    bps_join, on=["kapanewon_code", "tahun"], how="left",
    validate="many_to_one", indicator=True)
n_after = len(integrated_hourly)

print(f"Baris sebelum join : {n_before:,}")
print(f"Baris sesudah join : {n_after:,}  (harus sama)")
print("Baris tanpa pasangan BPS:", (integrated_hourly["_merge"] == "left_only").sum())
print("Duplikat (kode, datetime):", integrated_hourly.duplicated(["kapanewon_code", "datetime"]).sum())
print("Baris dengan n_maids kosong:", integrated_hourly["n_maids"].isna().sum())

integrated_hourly = integrated_hourly.drop(columns="_merge")

Baris sebelum join : 230,418
Baris sesudah join : 230,418  (harus sama)
Baris tanpa pasangan BPS: 0
Duplikat (kode, datetime): 0
Baris dengan n_maids kosong: 0


In [9]:
hourly_f = integrated_hourly.copy()
hourly_f["is_night"] = hourly_f["hour"].isin([22, 23, 0, 1, 2, 3, 4, 5])

agg = (hourly_f.groupby(["kapanewon_code", "tahun"])
    .agg(n_slot_terbit=("n_maids", "size"),
         mean_n_maids=("n_maids", "mean"),
         mean_share_in_hour=("share_in_hour", "mean"),
         total_pings=("n_pings", "sum"))
    .reset_index())

night = (hourly_f[hourly_f["is_night"]]
    .groupby(["kapanewon_code", "tahun"])["n_maids"].mean()
    .rename("mean_n_maids_malam").reset_index())

integrated_kapanewon = (dim
    .merge(bps[["kapanewon_code", "tahun", "total", "laki_laki", "perempuan"]]
           .rename(columns={"total": "pop_bps", "laki_laki": "pop_bps_laki",
                            "perempuan": "pop_bps_perempuan"}),
           on="kapanewon_code", how="left", validate="one_to_many")
    .merge(agg, on=["kapanewon_code", "tahun"], how="left", validate="one_to_one")
    .merge(night, on=["kapanewon_code", "tahun"], how="left", validate="one_to_one")
    .merge(hm[["kapanewon_code", "n_users", "ratio", "ratio_gt_1"]],
           on="kapanewon_code", how="left", validate="many_to_one"))

integrated_kapanewon["maids_per_1000_penduduk"] = (
    integrated_kapanewon["mean_n_maids"] / integrated_kapanewon["pop_bps"] * 1000)

print(integrated_kapanewon.shape, "(harapan: 156 baris)")
print("Duplikat (kode, tahun):", integrated_kapanewon.duplicated(["kapanewon_code", "tahun"]).sum())
print("Kapanewon tanpa data kehadiran:", integrated_kapanewon["mean_n_maids"].isna().sum())
integrated_kapanewon.head()

(156, 17) (harapan: 156 baris)
Duplikat (kode, tahun): 0
Kapanewon tanpa data kehadiran: 0


,kapanewon_code,kab_kota,kapanewon,kab_code,tahun,pop_bps,pop_bps_laki,pop_bps_perempuan,n_slot_terbit,mean_n_maids,mean_share_in_hour,total_pings,mean_n_maids_malam,n_users,ratio,ratio_gt_1,maids_per_1000_penduduk
0,34.01.01,KULON PROGO,TEMON,34.01,2021,28437,14159,14278,743,51.864065,0.006015,120032,26.623482,2330,0.081927,False,1.823823
1,34.01.01,KULON PROGO,TEMON,34.01,2022,28673,14274,14399,2352,25.528912,0.007227,153502,16.333879,2330,0.081927,False,0.890347
2,34.01.02,KULON PROGO,WATES,34.01,2021,48888,24317,24571,744,120.385753,0.013521,171632,62.903226,7250,0.148292,False,2.462481
3,34.01.02,KULON PROGO,WATES,34.01,2022,49101,24421,24680,2507,54.082569,0.011307,287552,37.921131,7250,0.148292,False,1.101456
4,34.01.03,KULON PROGO,PANJATAN,34.01,2021,38385,18990,19395,677,14.706056,0.001706,30742,9.912568,1040,0.027090,False,0.38312


In [10]:
integrated_hourly.to_parquet(OUT / "integrated_hourly.parquet", index=False)
integrated_kapanewon.to_parquet(OUT / "integrated_kapanewon.parquet", index=False)

db = OUT / "jogjapulse.duckdb"
con = duckdb.connect(str(db))
tabel_db = {
    "dim_kapanewon": dim,
    "bps_population": bps,
    "fact_presence_hourly": integrated_hourly,
    "integrated_kapanewon": integrated_kapanewon,
    "od_home_visits_monthly": pd.read_parquet(CLEAN / "od_home_visits_kapanewon_monthly.parquet"),
    "od_origin_homes_monthly": pd.read_parquet(CLEAN / "od_origin_homes_monthly.parquet"),
    "fact_presence_cell500m_daily": pd.read_parquet(CLEAN / "presence_cell500m_daily.parquet"),
}
for nama, df in tabel_db.items():
    con.register("tmp_df", df)
    con.execute(f"CREATE OR REPLACE TABLE {nama} AS SELECT * FROM tmp_df")
    con.unregister("tmp_df")

print(con.execute("SHOW TABLES").fetchall())
con.close()

[('bps_population',), ('dim_kapanewon',), ('fact_presence_cell500m_daily',), ('fact_presence_hourly',), ('integrated_kapanewon',), ('od_home_visits_monthly',), ('od_origin_homes_monthly',)]


In [12]:
print(integrated_hourly.groupby("supply_regime")["date"].agg(["min", "max", "size"]))

                     min        max   size
supply_regime                             
A             2021-12-01 2021-12-31  54867
B             2022-01-01 2022-02-15  80837
C             2022-02-16 2022-04-30  94714


In [13]:
agg_regime = (integrated_hourly.assign(is_night=integrated_hourly["hour"].isin([22,23,0,1,2,3,4,5]))
    .groupby(["kapanewon_code", "supply_regime"])
    .agg(n_slot_terbit=("n_maids", "size"),
         mean_n_maids=("n_maids", "mean"),
         mean_share_in_hour=("share_in_hour", "mean"),
         total_pings=("n_pings", "sum"))
    .reset_index())

# Populasi BPS 2022 dipakai sebagai pembanding regime di tahun 2022
pop22 = bps[bps["tahun"] == 2022][["kapanewon_code", "total"]].rename(columns={"total": "pop_bps_2022"})
agg_regime = agg_regime.merge(pop22, on="kapanewon_code", how="left", validate="many_to_one")
agg_regime = agg_regime.merge(dim, on="kapanewon_code", how="left", validate="many_to_one")
print(agg_regime.shape, "| tanpa pop:", agg_regime["pop_bps_2022"].isna().sum())

(234, 10) | tanpa pop: 0
